In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
import torch.nn as nn
import numpy as np
import pickle
import wandb
import timm
import time
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device('cuda')
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Free   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1024**3:.1f}GB")

# Load config
with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

train_df      = config['train_df']
val_df        = config['val_df']
test_df       = config['test_df']
CLASSES       = config['classes']
class_weights = config['class_weights']
IMAGE_SIZE    = config['image_size']
SEG_DIR       = '../data/segmented_images'

print(f"Train  : {len(train_df):,}")
print(f"Classes: {len(CLASSES)}")
print("✅ Ready!")

/home/lhotse1/student/btech2023/achanta/miniconda3/envs/cv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU    : Tesla V100-PCIE-32GB
Free   : 31.7GB
Train  : 80,726
Classes: 15
✅ Ready!


In [9]:
# Build ConvNeXt Base model
model = timm.create_model(
    'convnext_base',
    pretrained=True,
    num_classes=len(CLASSES),
    drop_rate=0.3,        # add dropout
    drop_path_rate=0.2    # add stochastic depth
)

model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ ConvNeXt Base loaded → {n_params:,} parameters")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 5e-5,
    weight_decay=1e-5
)

criterion = nn.BCEWithLogitsLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

print("✅ Model ready!")

✅ ConvNeXt Base loaded → 87,581,839 parameters
✅ Model ready!


In [10]:
import os
os.environ['HF_TOKEN'] = 'REDACTED_HF_TOKEN' 

In [11]:
FILENAME_COL = 'Image Index'
LABEL_COL    = 'Finding Labels'

def build_multihot(df, classes):
    label_lists  = df[LABEL_COL].str.split('|')
    multihot     = np.zeros((len(df), len(classes)), dtype=np.float32)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for row_idx, labels in enumerate(label_lists):
        for lbl in labels:
            lbl = lbl.strip()
            if lbl in class_to_idx:
                multihot[row_idx, class_to_idx[lbl]] = 1.0
    return multihot

class NIHDataset(Dataset):
    def __init__(self, df, image_dir, classes, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.labels    = build_multihot(df, classes)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx][FILENAME_COL]
        img   = Image.open(os.path.join(self.image_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = NIHDataset(train_df, SEG_DIR, CLASSES, train_transform)
val_dataset   = NIHDataset(val_df,   SEG_DIR, CLASSES, val_transform)
test_dataset  = NIHDataset(test_df,  SEG_DIR, CLASSES, val_transform)

# Weighted sampler
weight_vec     = np.array([class_weights[c] for c in CLASSES])
sample_weights = (train_dataset.labels * weight_vec).max(axis=1)
sampler        = WeightedRandomSampler(sample_weights,
                                       len(sample_weights),
                                       replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32,
                          sampler=sampler, num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32,
                          shuffle=False,  num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32,
                          shuffle=False,  num_workers=8, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")
print("✅ DataLoaders ready!")

Train batches : 2523
Val batches   : 281
Test batches  : 701
✅ DataLoaders ready!


In [13]:
import wandb
import time

wandb.init(
    project='xray-classification',
    name='03-classification-ConvNeXt',
    config={
        'model': 'ConvNeXt-Base',
        'image_size': IMAGE_SIZE,
        'batch_size': 32,
        'lr': 1e-4,
        'num_classes': len(CLASSES)
    }
)

os.makedirs('../checkpoints/ConvNeXt', exist_ok=True)
CHECKPOINT_PATH = '../checkpoints/ConvNeXt/latest.pt'
BEST_PATH       = '../checkpoints/ConvNeXt/best.pt'

NUM_EPOCHS           = 50
EARLY_STOP_PATIENCE  = 7
start_epoch          = 0
best_val_auc         = 0.0
epochs_since_improvement = 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch              = ckpt['epoch'] + 1
    best_val_auc             = ckpt['best_val_auc']
    epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
    print(f"🔄 Resumed from epoch {start_epoch} (best AUC: {best_val_auc:.4f})")
else:
    print("🆕 Starting fresh!")

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="Train" if training else "Val"):
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_preds.append(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    aucs = [roc_auc_score(all_labels[:, i], all_preds[:, i])
            for i in range(len(CLASSES)) if all_labels[:, i].sum() > 0]
    return total_loss / len(loader), np.mean(aucs)

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_auc = run_epoch(train_loader, training=True)
    val_loss,   val_auc   = run_epoch(val_loader,   training=False)
    scheduler.step(val_auc)
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | "
          f"Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} | "
          f"{elapsed:.0f}s")

    wandb.log({'epoch': epoch+1, 'train_loss': train_loss,
               'train_auc': train_auc, 'val_loss': val_loss,
               'val_auc': val_auc})

    if val_auc > best_val_auc:
        best_val_auc             = val_auc
        epochs_since_improvement = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_auc': val_auc}, BEST_PATH)
        print(f"  💾 Best saved (AUC: {val_auc:.4f})")
    else:
        epochs_since_improvement += 1

    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_auc': best_val_auc,
        'epochs_since_improvement': epochs_since_improvement
    }, CHECKPOINT_PATH)

    if epochs_since_improvement >= EARLY_STOP_PATIENCE:
        print("⏹️ Early stopping!")
        break

wandb.log({'final_best_val_auc': best_val_auc})
print(f"\n✅ Done! Best AUC: {best_val_auc:.4f}")

epoch,▁
final_best_val_auc,▁
train_auc,▁
train_loss,▁
val_auc,▁
val_loss,▁
epoch,9
final_best_val_auc,0.78528
train_auc,0.9866
train_loss,0.06882
val_auc,0.74613


🆕 Starting fresh!


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:32<00:00,  8.60it/s]


Epoch 1/50 | Train Loss: 0.0552 AUC: 0.9905 | Val Loss: 0.3176 AUC: 0.7383 | 1280s
  💾 Best saved (AUC: 0.7383)


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:32<00:00,  8.68it/s]


...

Epoch 11/50 | Train Loss: 0.0334 AUC: 0.9954 | Val Loss: 0.3364 AUC: 0.7409 | 1271s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:32<00:00,  8.60it/s]


Epoch 12/50 | Train Loss: 0.0334 AUC: 0.9953 | Val Loss: 0.3352 AUC: 0.7402 | 1271s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:32<00:00,  8.59it/s]


Epoch 13/50 | Train Loss: 0.0333 AUC: 0.9955 | Val Loss: 0.3348 AUC: 0.7409 | 1273s
⏹️ Early stopping!

✅ Done! Best AUC: 0.7430
